# Chapter 19: Experience Rating, Retrospective Rating and Bonus-Malus

Code of Chapter 19 of *Practical Introduction to Non-Life Insurance Actuarial Mathematics* (F.-X. Cart-Tanneur). Run the cells from the top: later cells reuse the data and imports of earlier ones. Each comment after `#` gives the value printed in the book.

In [ ]:
# Setup: make the book's modules importable (works locally and in the browser)
import os, sys
sys.path.insert(0, os.path.abspath(os.path.join("..", "src")))

The walkthrough comes first, because it loads the data and imports used by the other examples of the chapter.

## 19.6 Python Walkthrough

In [ ]:
import numpy as np
from scipy import optimize
from gidata import synthetic_portfolio
from gilossmodels import fit_mle
from giexperience import (experience_mod, split_plan_mod, retro_premium, simulate_fleet_losses,
                          bm_transition, stationary, mean_relativity, efficiency, simulate_bm)

# Section 19.1
experience_mod(35_000, 50_000, 0.25)                                   # 0.925
split_plan_mod([1200, 3500, 8000, 22000, 600, 4000], 5000, 27_000, 18_000, 0.3, 20_000)  # 0.8908

# Section 19.2: retro plan on a simulated fleet
_, claims = synthetic_portfolio()
L9 = fit_mle(claims["ultimate_amount"].to_numpy(), "lognormal")
mu, sigma = L9["mu"], L9["sigma"]
L = simulate_fleet_losses(100_000, 25, mu, sigma)
R = retro_premium(L, 26_000, 1.10, 1.04, 78_000, 182_000)
R.mean(), (R == 78_000).mean(), (R == 182_000).mean()                  # 136512, 0.031, 0.164
optimize.brentq(lambda b: retro_premium(L, b, 1.10, 1.04, 78_000, 182_000).mean() - 130_000,
                -50_000, 50_000)                                       # 18232

# Section 19.3: bonus-malus
rel = [0.60, 0.70, 0.80, 1.00, 1.20, 1.50]
stationary(bm_transition(0.10, 6, 2)).round(4)                         # 0.7829 ... 0.0051
mean_relativity(0.10, rel, 2), efficiency(0.10, rel, 2)                # 0.650, 0.110
simulate_bm(0.10, 6, 2, start=3, years=20, n_drivers=20_000).iloc[[4, 19]]

## Exercise 2: A retro premium in a bad year

In [ ]:
L = simulate_fleet_losses(100_000, 25, mu, sigma)
(retro_premium(L, 26_000, 1.10, 1.04, 78_000, 182_000) == 182_000).mean()   # 0.164

## Exercise 3: A steeper malus

In [ ]:
rel = [0.60, 0.70, 0.80, 1.00, 1.20, 1.50]
mean_relativity(0.10, rel, up_per_claim=3), efficiency(0.10, rel, up_per_claim=3)   # 0.700, 0.175